# ☁️ Climate Intervention in a Cloud Model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jslim93/DropLab/blob/main/notebooks/Climate_Intervention.ipynb)
&nbsp; *No install? Click the badge to run this in Google Colab — you only need a
Google account. (Run the **Colab setup** cell first; it takes ~1 min.)*

### Can we deliberately change a cloud — to cool the planet, or to make it rain?

This notebook is a guided tour of **aerosol–cloud–radiation coupling** using
droplab, an educational 2-D Lagrangian model of marine **stratocumulus** — the low,
flat cloud sheets that blanket cool oceans and act as one of Earth's biggest
sunshades.

We will run the *same model* five times to tell one story:

| # | Experiment | The lever we pull | What to watch |
|---|------------|-------------------|---------------|
| 1 | **The deck** | (nothing — baseline) | A realistic Sc cloud forms |
| 2 | **Marine Cloud Brightening** | spray *tiny* sea-salt | droplets shrink → cloud brightens → **cooling** |
| 3 | **Precipitation seeding** | inject a few *giant* CCN | rain is triggered |
| 4 | **Clean vs polluted** | change background aerosol | dirty air ⇒ bright; clean air ⇒ drizzly |
| 5 | **Entrainment mixing** | change *how* dry air mixes in | fewer, larger drops; dimmer; drizzlier |

> ### The one idea to take away
> **A cloud is not a fixed object — it is an *aerosol-mediated state*.** The same
> air can host a bright, long-lived, non-raining cloud *or* a dim, drizzly,
> short-lived one, depending on how many aerosol particles seed its droplets.
> Marine cloud brightening and rain seeding are the same lever pulled in
> **opposite directions**.


## How to use this notebook — Predict → Observe → Explain → Refine

This is a *laboratory*, not a slideshow. For each experiment:

1. **Predict** — Before running a cell, read the *🤔 Predict* prompt and commit to
   an answer (out loud or on paper). Predicting first is one of the most effective
   ways to learn from a simulation (the predict–observe–explain approach). Watch for
   **⚠️ "Common wrong intuition" (M#)** flags: these name a belief many people hold so
   you can test it head-on.
2. **Observe** — Run the cell and read the figure and the printed numbers.
3. **Explain** — Read the *📖 Explain* note and check it against **your own
   prediction**. If they disagree, that gap is exactly where the learning happens.
   Open the **✅ Check yourself** boxes to test whether the idea really stuck.
4. **Refine** — In Experiment 6 and the live panel, *change one thing and re-run* to
   hit a target or chase a "what if?". Inverting the physics to reach a goal is where
   understanding becomes usable.

### ⏱️ A note on runtime (important)
Each simulation is real physics, not a cartoon, so cells take time:

* Most cells run in **15–50 s**.
* **Experiment 2 (brightening) is the slow one — about 2 minutes** — because the
  cooling signal only emerges once the injected aerosol is lofted into the cloud
  and the deck re-equilibrates. It is the scientific headline, so it earns the wait.
* **Surface rain needs a long spin-up.** Brightness reacts almost instantly, but
  rain must *grow and fall*. So we only trust surface-rain numbers in the
  dedicated long runs (Experiments 2–3); for the quick runs we read **droplet
  size and albedo**, which are robust.


In [ ]:
# --- Colab setup (zero-install): on Google Colab this clones + installs DropLab;
# everywhere else it does nothing. Run me first on Colab, then continue normally. ---
import sys, os
if "google.colab" in sys.modules:
    if not os.path.isdir("DropLab"):
        !git clone --depth 1 https://github.com/jslim93/DropLab.git
    %cd DropLab
    !pip install -q -e .
    print("Colab ready. If an import fails, use Runtime > Restart session, then re-run.")
else:
    print("Not on Colab — nothing to do; run the next cell.")


In [ ]:
# --- setup: import the model API and the packaged demos (read-only consumers) ---
import sys, os
# Make the droplab package importable whether the kernel starts in the repo root
# or in this notebooks/ subfolder (Jupyter sets the kernel's CWD to the notebook's
# directory, so we look here and one level up for the droplab/ package).
for _cand in (os.getcwd(), os.path.abspath(os.path.join(os.getcwd(), os.pardir))):
    if os.path.isdir(os.path.join(_cand, "droplab")) and _cand not in sys.path:
        sys.path.insert(0, _cand)

import time
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import HTML

# The droplab climate API
from droplab.climate_widget import simulate, figure, climate_panel
from droplab.climate_diag import column_optics, twomey_report, toa_forcing
from droplab.flow2d import Flow2D

# The packaged demo scripts (we import and CALL them; we never modify them)
import examples.mcb_demo as mcb
import examples.precip_seeding as precip
import examples.precip_vs_nonprecip as pvn
import examples.entrainment_mixing as ent

# The example modules switch matplotlib to the headless "Agg" backend on import;
# restore inline rendering so figures show up in the notebook.
%matplotlib inline
plt.rcParams["animation.embed_limit"] = 80  # MB, for the embedded animation

def reff_albedo(res):
    # Quick domain-mean (effective radius in micron, cloud albedo) from a run dict.
    o = column_optics(res["M"], res["A"], res["x"], res["z"], res["flow"])
    return o["reff_mean"] * 1e6, o["albedo_mean"]

print("droplab climate API loaded. Ready.")


---
## 1 · The marine stratocumulus deck

Our laboratory cloud is a **DYCOMS-II** marine stratocumulus — a textbook research
case. Two ingredients make it realistic:

* a **sharp inversion** (warm, dry air sitting on top of cool, moist boundary-layer
  air) that traps the cloud in a thin layer, and
* **cloud-top radiative cooling**: the cloud top radiates heat to space, cools,
  becomes dense, and sinks — this is the engine that stirs the whole boundary layer
  and keeps the deck alive.

The boundary is **periodic** in the horizontal, so the deck behaves like an endless
sheet rather than a single isolated puff.

> ### 🤔 Predict
> With a *moderately clean* background of **N₀ = 120 particles/cm³**, do you expect
> the droplets to be small (~5 µm) or large (~15 µm)? Will the cloud be bright
> (albedo > 0.5) or dim?


In [ ]:
# Baseline deck: no intervention. A moderately clean marine background.
t0 = time.time()
res0, sum0 = simulate(background_N=120.0, ihmd=0.0, seed_on=False, nt=700)
print(f"ran in {time.time()-t0:.0f} s")
print(f"effective radius : {sum0['reff_um']:.1f} µm")
print(f"cloud albedo     : {sum0['albedo']:.3f}")
figure(res0, sum0, title="Baseline marine stratocumulus deck (N0=120/cc, no seeding)");


> ### 📖 Explain
> You should see a coherent cloud layer near the inversion with droplets of order
> ~10 µm and an albedo around 0.4–0.6 — a recognisable marine Sc deck. The droplet
> dots are coloured by size; this is the **canvas** on which we now intervene.


---
## 2 · Marine Cloud Brightening (MCB) — cooling by making droplets smaller

**The idea:** A fleet of ships sprays a fine mist of **sub-micron sea-salt** into the
boundary layer. Each salt particle is an extra cloud-condensation nucleus (CCN), so
the *same* cloud water is shared among **more, smaller** droplets.

**Why that cools the planet (the Twomey effect):** more, smaller droplets present
*more total surface area* to sunlight, so the cloud reflects more — *with no change
in how much water it holds*. A brighter cloud over a dark ocean = less sunlight
absorbed = a cooling at the top of the atmosphere.

We run the deck **twice with identical weather** (same turbulence seed): an unseeded
control and a seeded run, then compare the *domain-mean* — because a turbulent cloud
is chaotic and the Twomey signal is only robust in the spatial average (this is why
real ship-tracks show up in *time-mean* satellite composites, not single snapshots).

> ### 🤔 Predict
> Sketch two histograms of droplet effective radius — *unseeded* and *seeded*. Which
> way does the seeded one shift? Which way does albedo move? Is the top-of-atmosphere
> forcing positive (warming) or negative (cooling)?
>
> ⚠️ **Common wrong intuition (M2):** *"More salt particles → bigger droplets."* It
> feels right, but it is backwards. Predict first, then watch what actually happens.

> ⏱️ *This is the ~2-minute cell.* It runs the full-resolution demo so the cooling
> signal is real and quotable.


In [ ]:
# Full MCB demo: unseeded vs seeded, identical meteorology. ~2 minutes.
t0 = time.time()
base_mcb, seed_mcb = mcb.run(nt=1500)
print(f"ran both decks in {time.time()-t0:.0f} s")

# Twomey diagnostics (domain-mean reff, optical depth, albedo, and TOA forcing)
flow = Flow2D(X=mcb.CFG["X"], Z=mcb.CFG["Z"], Nx=mcb.CFG["Nx"], Nz=mcb.CFG["Nz"])
rep = twomey_report(base_mcb, seed_mcb, flow)
print(f"  Δ effective radius : {rep['d_reff_um']:+.2f} µm   (negative = smaller drops)")
print(f"  Δ cloud albedo     : {rep['d_albedo']:+.4f}        (positive = brighter)")
print(f"  TOA forcing        : {rep['forcing_wm2']:+.2f} W/m²  (negative = COOLING)")

# The Twomey fingerprint: distributions of reff and albedo, unseeded vs seeded
mcb.figure(base_mcb, seed_mcb);


Now watch it happen in time. The animation shows the seeded deck (seeded droplets
ringed in magenta) beside the **live domain-mean albedo** and the **accumulated
cooling** since injection.

In [ ]:
# Animation: seeded deck + albedo curve + accumulated TOA cooling (no new run)
from droplab.flow2d_viz import animate_mcb
fig_a, anim = animate_mcb(base_mcb, seed_mcb,
                          t_inject=mcb.MCB_SEED["t_inject"],
                          dt=mcb.CFG["dt"], fps=8, r_max=40.0)
plt.close(fig_a)            # show only the animation, not a static duplicate
HTML(anim.to_jshtml())


> ### 📖 Explain — *compare this with your prediction*
> The seeded histogram shifts to **smaller radius** and **higher albedo**, and the
> TOA forcing is **negative (a cooling)** — for this deck of order a few W/m². That is
> marine cloud brightening: a microphysical nudge (more CCN) becomes a radiative
> lever (a brighter cloud), with the cloud's water content essentially unchanged.
>
> ⚠️ **Read the W/m² as a model output, not a forecast.** It comes from a *slab-albedo*
> radiation estimate over this idealized 2-D deck and varies with the turbulence seed
> and resolution. The durable, transferable result is the **sign and mechanism**
> (more CCN → smaller drops → brighter → cooling) — not the exact number. Real-world
> MCB forcing is far more uncertain and regionally variable than one toy column.
>
> If you predicted *bigger* drops (M2), this is the moment to update: the **same
> water** split among **more nuclei** makes **more, smaller** droplets — and smaller
> droplets, having more total surface area, reflect more sunlight.
>
> **Linking the views:** the magenta-ringed dots in the animation are the *same*
> injected droplets whose smaller sizes produce the left-shifted radius histogram,
> which in turn is what lifts the albedo curve and accumulates the cooling. One
> intervention, read three ways.
>
> **Caveat to keep honest:** the per-column signal is noisy because seeding also
> nudges the turbulence; only the *domain-mean* shift is trustworthy.

<details>
<summary><b>✅ Check yourself</b> — click to reveal</summary>

> **Q:** You double the injected sea-salt. Does cloud liquid water content go up,
> down, or stay roughly the same — and does albedo still rise?
>
> **A:** Liquid water stays **roughly the same** (the air holds about as much
> condensate either way); albedo still **rises**, because the *same* water is divided
> among yet more, even smaller droplets. Brightening is a redistribution of water
> across droplet sizes, not an addition of water.
</details>


---
## 3 · Precipitation seeding — tipping a marginal cloud into rain

Now we pull the lever the *other* way. Instead of many tiny particles, we inject a
**few giant CCN** (~1.5 µm dry sea-salt). Because of their large solute mass they
activate instantly and grow fast, becoming the first **drizzle drops**. As they fall
they collect smaller droplets (collision–coalescence) and kick-start a rain the
cloud was already on the edge of producing.

The deck here sits **right at its precipitation threshold** (background N ≈ 160/cm³,
barely drizzling) — the only regime where a small seed can actually tip the balance.

> ### 🤔 Predict
> Will surface rain go up a little, a lot, or not at all? And here is the subtle
> question: if rain *does* increase, is that the **cloud** raining, or just the heavy
> **seed drops** falling back out?
>
> ⚠️ **Common wrong intuitions to test here:**
> - **(M3)** *"Condensation makes rain."* In a warm cloud, condensation alone gets you
>   ~10 µm droplets and then stalls — it takes **collision–coalescence** to bridge to
>   millimetre raindrops.
> - **(M6)** *"A handful of giant particles is negligible."* Watch whether a *tiny*
>   dose of giant CCN changes the outcome at all.

> ⏱️ *Long run (~70 s).* Surface rain needs time to grow and fall — a short run would
> show **nothing**, which is itself a lesson about precipitation.


In [ ]:
# Precipitation seeding: marginal deck (N=160) with vs without giant CCN. ~70 s.
t0 = time.time()
base_p, seed_p = precip.run(nt=1700)
print(f"ran both decks in {time.time()-t0:.0f} s")

seed_mass = precip._seed_mass()                       # mass of the injected nuclei
ratio_rain = seed_p["surf_precip"] / max(base_p["surf_precip"], 1e-30)
ratio_seed = seed_p["surf_precip"] / seed_mass
print(f"  unseeded surface rain : {base_p['surf_precip']:.3e} kg")
print(f"  seeded surface rain   : {seed_p['surf_precip']:.3e} kg   ({ratio_rain:.1f}x more)")
print(f"  rain / injected seed mass = {ratio_seed:.0f}x   (>>1 ⇒ the CLOUD rained, not the seeds)")

precip.figure(base_p, seed_p);


> ### 📖 Explain — *compare this with your prediction*
> Surface rain roughly **quadruples**, and the precipitated mass is **~70× the
> injected seed mass** — proof that almost all the rain is *cloud* water the seeds
> triggered, not the seeds themselves falling out. That ratio is the honest test of
> "real" seeding (so much for M6: a *tiny* dose of giant CCN was decisive).
>
> And notice what the seeds actually *did*: they did not "condense harder." They
> became the first big drops that **collided with and collected** smaller droplets —
> that is the collision–coalescence bridge (M3), and because collection is *stochastic*
> the drop spectrum **broadens** into a few large rain embryos rather than everyone
> growing uniformly (M5).
>
> ### Two honest caveats (please remember these)
> 1. **20 µm "seeding" is cheating.** If you injected large drops directly, you would
>    get "rain" that is just the **seed mass falling out** — not the cloud raining.
>    Real seeding uses *small dry* giant CCN that must grow *inside* the cloud.
> 2. **You cannot tip a deck that is far from threshold.** A heavily polluted,
>    non-raining cloud will *not* be tipped into rain by a few nuclei — the seeds
>    grow to drizzle size but no deck-wide rain follows. Seeding only works on clouds
>    already poised at the edge.

<details>
<summary><b>✅ Check yourself</b> — click to reveal</summary>

> **Q:** If you ran this *exact* seeding on a heavily polluted deck (background
> N ≈ 400/cm³) instead of the marginal N ≈ 160 deck, would you expect ~4× the rain?
>
> **A:** **No.** That deck is far from its precipitation threshold (caveat 2): the
> giant CCN would grow to drizzle size but no deck-wide rain would follow. Seeding is
> a *nudge over an edge*, not a *create-rain-from-nothing* button. (You can test this
> in the interactive panel below.)
</details>


---
## 4 · Clean vs polluted decks — why dirty air makes brighter, drier clouds

Experiments 2–3 *added* aerosol on purpose. But the *background* aerosol already
sets the cloud's personality. Here we compare two decks with identical weather:

* a **clean** maritime deck (N ≈ 20/cm³) — few, **large** droplets, and
* a **polluted** continental deck (N ≈ 400/cm³) — many, **small** droplets.

> ### 🤔 Predict
> Which deck is **brighter**? Which one is more likely to **drizzle itself away**?
> (Hint: large drops fall; small drops float.)
>
> ⚠️ **Common wrong intuition (M2), again — naturally this time:** *"Dirtier air must
> make bigger raindrops."* You met this in Experiment 2 as a deliberate intervention;
> here it is the **background** aerosol doing the same thing. Predict the droplet sizes
> for clean vs polluted before you run.


In [ ]:
# Clean (N~20) vs polluted (N~400), identical meteorology. ~50 s.
t0 = time.time()
cases = pvn.run(nt=900)
print(f"ran in {time.time()-t0:.0f} s\n")
print(f"{'deck':24s}{'reff [µm]':>12s}{'albedo':>10s}")
for label, sub, r in cases:
    re, al = reff_albedo(r)
    print(f"{label:24s}{re:>12.1f}{al:>10.3f}")
pvn.figure(cases);


> ### 📖 Explain — *compare this with your prediction*
> The **polluted** deck has much **smaller** droplets (~8 µm) and a far **higher
> albedo** (~0.65) — it is bright and rain-suppressed. The **clean** deck makes
> **large** droplets (~30+ µm) that are dim and prone to drizzle. So M2 is wrong in
> both directions: more aerosol → *smaller* drops, whether you spray it in (Exp 2) or
> it is simply the background air (here). This is the natural, always-on version of
> the Twomey effect — and a reminder that *air pollution brightens low clouds*, a real
> and uncomfortable climate complication.
>
> *(We read **reff and albedo** here, which are robust at this run length; the
> clean deck's full drizzle signature needs the longer spin-up from Experiment 3.)*

<details>
<summary><b>✅ Check yourself</b> — click to reveal</summary>

> **Q:** A factory upwind doubles the aerosol over a marine deck. Name the *two*
> climate-relevant things that happen to the cloud — and say which one is a *warming*
> and which a *cooling* influence.
>
> **A:** (1) The cloud gets **brighter** (Twomey) → reflects more sunlight → a
> **cooling**. (2) It **rains less** and so tends to **last longer / hold more water**
> (the cloud-lifetime effect) → also generally a **cooling**. Both push the same way
> here — which is exactly why aerosol–cloud interactions are the largest *uncertainty*
> in estimates of human climate forcing.
</details>


---
## 5 · Entrainment mixing (IHMD) — *how* dry air invades matters

Stratocumulus constantly **entrains** dry air from above the inversion. *How* that dry
air evaporates the cloud droplets has two extremes:

* **Homogeneous mixing (IHMD = 0):** the dry air mixes instantly everywhere, so
  *every* droplet shrinks a little, but they all survive.
* **Inhomogeneous mixing (IHMD = 1):** pockets of dry air evaporate *some droplets
  completely* while leaving the rest at their original size — so you end up with
  **fewer, larger** droplets.

The model encodes this with a single knob: `N/N₀ = (q/q₀)^IHMD`.

> ### 🤔 Predict
> Going from IHMD = 0 to IHMD = 1, what happens to droplet **number**, droplet
> **size**, cloud **albedo**, and **drizzle**?
>
> ⚠️ **Common wrong intuition (M7):** *"Entrainment just dilutes the cloud uniformly —
> every droplet shrinks a bit."* That is only one extreme (IHMD = 0). Predict what the
> *other* extreme (IHMD = 1) does to droplet **number** vs **size** before you run.


In [ ]:
# Homogeneous (IHMD=0) vs inhomogeneous (IHMD=1) entrainment mixing. ~35 s.
t0 = time.time()
homo, inho = ent.run(nt=900)
print(f"ran in {time.time()-t0:.0f} s\n")
print(f"{'mixing':22s}{'ΣA (drops)':>14s}{'reff [µm]':>12s}{'albedo':>10s}{'rain [kg]':>12s}")
for label, r in [("IHMD=0 homogeneous", homo), ("IHMD=1 inhomogeneous", inho)]:
    re, al = reff_albedo(r)
    print(f"{label:22s}{r['A'].sum():>14.2e}{re:>12.2f}{al:>10.3f}{r['surf_precip']:>12.2e}")
ent.figure(homo, inho);


> ### 📖 Explain — *compare this with your prediction*
> Inhomogeneous mixing leaves **fewer droplets**, makes them **larger**, drops the
> **albedo** (dimmer cloud), and lets **more drizzle** form — exactly the chain
> *fewer/larger drops → dimmer → drizzlier*. So M7 is only half-true: homogeneous
> mixing (IHMD = 0) does shrink everyone uniformly, but inhomogeneous mixing (IHMD = 1)
> instead **removes whole droplets while the survivors keep their size**. The *style*
> of mixing, not just the *amount* of entrainment, changes the cloud's brightness and
> its rain. This is a genuinely unsettled question in cloud physics, and here you can
> feel why it matters.

<details>
<summary><b>✅ Check yourself</b> — click to reveal</summary>

> **Q:** Two parcels lose the *same* amount of cloud water to entrainment, one by
> homogeneous mixing and one by inhomogeneous mixing. Which ends up **brighter**, and
> why?
>
> **A:** The **homogeneous** parcel is brighter: it keeps all its droplets (just
> slightly smaller), so droplet *number* — which controls albedo — is preserved. The
> inhomogeneous parcel loses whole droplets, so fewer, larger drops make it dimmer and
> more prone to drizzle, even though both lost identical water.
</details>


---
## 6 · 🎯 Challenge — hit a target with **one** knob

You have met each lever on its own. Now *aim*. This is the deliberate middle step
between fully-worked experiments and open free-play: **everything is fixed except one
control**, and you have a **target to reach**. Adjust, re-run, and close in — this is
the *Refine* in Predict → Observe → Explain → **Refine**.

> ### 🎯 Your mission
> Using **only `my_N`** in the cell below, find the background aerosol that makes the
> deck's **albedo land within 0.55 ± 0.02**.
>
> **Predict first:** from Experiment 4 (clean N≈20 → albedo ~0.33; polluted N≈400 →
> ~0.66), do you expect the answer nearer **50, 150, or 350**? Commit to a guess, then
> test and refine.


In [ ]:
# Change ONLY this number, re-run the cell, and try to hit albedo 0.55 +/- 0.02.
my_N = 150.0          # <-- your guess, particles/cm^3   (clean ~20, polluted ~400)

TARGET, TOL = 0.55, 0.02
_res, _sum = simulate(background_N=my_N, ihmd=0.0, seed_on=False, nt=500)
alb = _sum["albedo"]
print(f"background_N = {my_N:.0f}/cm3   ->   albedo = {alb:.3f}    (target {TARGET} +/- {TOL})")
if abs(alb - TARGET) <= TOL:
    print("HIT! You dialed the cloud to this brightness with a single knob.")
elif alb < TARGET:
    print("Too DIM  -> the cloud needs MORE, smaller droplets: RAISE my_N and re-run.")
else:
    print("Too BRIGHT -> fewer, larger droplets: LOWER my_N and re-run.")


> ### 📖 Explain
> Notice the *feedback you used*: "too dim → more aerosol", "too bright → less". You
> were running the Twomey relationship **backwards** — inverting a physical law to hit
> a goal. That is exactly the reasoning a real marine-cloud-brightening controller
> would need, and it is why understanding the *mechanism* (not just memorising "more
> aerosol = brighter") matters.


---
## 🎛️ Your turn — open exploration with the control panel

Everything above is now behind **four knobs and a Run button**. Change the background
aerosol, the entrainment-mixing degree, and whether/how you seed the cloud, then press
**▶ Run simulation** and read the effective radius, albedo, and rain.

**Experiments to try:**
* Push **background aerosol** from 20 → 500 and watch albedo climb (the Twomey effect,
  live).
* Turn on **MCB sea-salt** seeding on a moderate deck — how negative can you make the
  forcing?
* Switch to **GCCN (precip)** seeding with a ~1.5 µm dry radius on a marginal deck —
  can you trigger rain?
* Slide **IHMD** from 0 → 1 and watch the droplets grow and the cloud dim.

*(Each run takes ~20–40 s. Longer runs let rain develop.)*


In [ ]:
climate_panel()

---
## Wrap-up — the cloud is a state, not an object

| Lever | Microphysics | Cloud response | Climate meaning |
|-------|--------------|----------------|-----------------|
| **+ tiny sea-salt** (MCB) | more, smaller drops | brighter, rain-suppressed | **cooling** (Twomey) |
| **+ giant CCN** (seeding) | a few big drops fall & collect | rain triggered | precipitation enhancement |
| **+ background aerosol** | more, smaller drops | brighter, drier | pollution brightens clouds |
| **inhomogeneous mixing** | fewer, larger drops | dimmer, drizzlier | a real modelling uncertainty |

**The honest limits of these interventions:**
* Marine cloud brightening's signal is only robust in the **domain mean**; per-column
  it is buried in turbulent noise.
* Precipitation seeding only works on a deck **near its threshold**, and "rain" must be
  the *cloud's* water, not the seed mass falling out.
* These are **idealised 2-D experiments** for building intuition — not operational
  forecasts of any real intervention.

### Where to go next
* **Browser app** — the same four knobs without any code:
  `streamlit run app/streamlit_climate.py`
* **Written guide** — `docs/CLIMATE_INTERVENTION.md`
* **Instructors** — session plans, answer keys, a question-authoring template, and a
  starter concept inventory live in `docs/CLIMATE_INTERVENTION_INSTRUCTOR.md`.
* **The demo scripts** you just called: `examples/mcb_demo.py`,
  `examples/precip_seeding.py`, `examples/precip_vs_nonprecip.py`,
  `examples/entrainment_mixing.py` (run any with, e.g., `python -m examples.mcb_demo`).
